# 09 — Feature-group ablation and locked final holdout

Five expanding **development-only** folds; newest 20% of timestamps are held out and **never scored in this notebook**. Compare baseline, L2, and L1 logistic classifiers across original, full, and leave-one-feature-group-out schemes. All feature preparation and labels are fitted per development fold.

In [ ]:
from pathlib import Path
from IPython.display import display
from stock_ml_dynamic.pooled_models import load_master
from stock_ml_dynamic.feature_ablation import run_ablation, plot_ablation, summarize_ablation, GROUPS
ROOT = next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pyproject.toml').is_file()), None)
if ROOT is None: raise RuntimeError('Open from within the repository')
INTERVAL='1d'
required=('master_labeled.parquet','latest_inference.parquet','transformers.joblib','manifest.json')
candidates=[ROOT/'artifacts'/'master_tables'/INTERVAL, ROOT/'notebooks'/'artifacts'/'master_tables'/INTERVAL]
MASTER=next((p for p in candidates if all((p/f).is_file() for f in required)),None)
if MASTER is None: raise FileNotFoundError(f'Run Notebook 03 first. Checked: {candidates}')
master,latest,bundle,manifest=load_master(MASTER)
print('Master:',MASTER,'| symbols:',master.symbol.nunique(),'| rows:',len(master))
display(GROUPS)

In [ ]:
N_FOLDS=5
HOLDOUT_FRACTION=0.20
MAX_TRAIN_ROWS=120_000
OUTPUT=ROOT/'artifacts'/'feature_ablation'/INTERVAL
metrics,curves,eligibility,holdout_start_boundary=run_ablation(
    master,n_folds=N_FOLDS,holdout_fraction=HOLDOUT_FRACTION,
    max_train_rows=MAX_TRAIN_ROWS,output_dir=OUTPUT)
print('Development ends:',holdout_start_boundary)
print('Final holdout: LOCKED; no final holdout model scores calculated')
display(eligibility.loc[~eligibility.eligible])
display(metrics.tail())

In [ ]:
summary=plot_ablation(metrics,curves,OUTPUT)
display(summary.round(5))
paired=metrics.pivot_table(index=['fold','model'],columns='scheme',values=['accuracy','balanced_accuracy','roc_auc_ovr_macro'])
for metric in ('accuracy','balanced_accuracy','roc_auc_ovr_macro'):
    for scheme in sorted(set(metrics.scheme)-{'all_expanded'}):
        paired[(metric,'gain_vs_'+scheme)]=paired[(metric,'all_expanded')]-paired[(metric,scheme)]
display(paired.round(5))
print('Saved development-only outputs:',OUTPUT)

## Decision gate

Select one **model + feature scheme** using only the development CV summary and consistency across five folds. Do **not** inspect final holdout outcomes until the configuration and hyperparameters are frozen. Once selected, build a separate one-time final evaluation with fold-specific preprocessing fitted on all eligible development history.

**Interpretation caveats:** leave-one-out peer-market returns at the current bar require all relevant peer closes to be available by prediction time; latest universe membership can cause survivorship bias; ablation correlations mean removing one group does not establish causal importance. This notebook uses the existing master table, so ensure it was generated with complete, timestamp-aligned prices. No neural networks, forests, or regressors are trained.